In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e12/sample_submission.csv
/kaggle/input/playground-series-s5e12/train.csv
/kaggle/input/playground-series-s5e12/test.csv


In [2]:
class CONFIG:
    INPUT_DIR = '/kaggle/input/playground-series-s5e12'
    
    N_FOLDS = 5
    SEED = 42

    TARGET = 'diagnosed_diabetes'

config = CONFIG()

train = pd.read_csv(f'{config.INPUT_DIR}/train.csv')
test = pd.read_csv(f'{config.INPUT_DIR}/test.csv')
# test[config.TARGET] = -1
train['source'] = 'train'
test['source']= 'test'

combine = pd.concat([train, test])
submission = pd.read_csv(f'{config.INPUT_DIR}/sample_submission.csv')

In [3]:
def eda(df, name):
    print(f"Exploring {name} dataframe")
    print('='*30)
    print(f'\n NULL VALUES: {df.isnull().sum()}')
    print('='*30)
    print(f'\n {name} dataframe shape: {df.shape}')
    print('='*30)
    print(f'\n {name} dataframe numerical stats: {df.describe()}')
    print('='*30)

eda(train, 'TRAIN')
eda(test, 'TEST')

Exploring TRAIN dataframe

 NULL VALUES: id                                    0
age                                   0
alcohol_consumption_per_week          0
physical_activity_minutes_per_week    0
diet_score                            0
sleep_hours_per_day                   0
screen_time_hours_per_day             0
bmi                                   0
waist_to_hip_ratio                    0
systolic_bp                           0
diastolic_bp                          0
heart_rate                            0
cholesterol_total                     0
hdl_cholesterol                       0
ldl_cholesterol                       0
triglycerides                         0
gender                                0
ethnicity                             0
education_level                       0
income_level                          0
smoking_status                        0
employment_status                     0
family_history_diabetes               0
hypertension_history                  0

In [4]:
FEATURES = [col for col in train.columns if col not in ['id', 'diagnosed_diabetes']]
CATS = train[FEATURES].select_dtypes(include='object').columns.to_list()
NUMS = train[FEATURES].select_dtypes(include=['int64', 'float64']).columns.to_list()

combine = pd.concat([train, test])

In [5]:
CATS1 = []

for c in CATS:
    n = f'{c}_enc'
    for df in [combine]:
        df[c] = df[c].astype('category')
        df[n], _ =  df[c].factorize()
        df[n] = df[n].astype('float32')

    CATS1.append(n)

In [6]:
combine[NUMS].nunique()

age                                    71
alcohol_consumption_per_week            9
physical_activity_minutes_per_week    578
diet_score                             99
sleep_hours_per_day                    69
screen_time_hours_per_day             152
bmi                                   234
waist_to_hip_ratio                     38
systolic_bp                            77
diastolic_bp                           54
heart_rate                             60
cholesterol_total                     170
hdl_cholesterol                        71
ldl_cholesterol                       165
triglycerides                         241
family_history_diabetes                 2
hypertension_history                    2
cardiovascular_history                  2
dtype: int64

In [7]:
# CATS2 = []
# SIZES = []

# for c in NUMS:
#     n = f'{c}_cat'
#     for df in [combine]:
        


In [8]:
train_idx = combine['source'] == 'train'
test_idx = combine['source'] == 'test'

train_n = combine[train_idx].reset_index(drop=True).copy()
test_n = combine[test_idx].reset_index(drop=True).copy()

In [9]:
FEATURES_T = CATS + CATS1 + NUMS
len(FEATURES_T)

32

In [10]:
X = train_n[FEATURES_T].copy()
y = train_n[config.TARGET]

test_n = test_n[FEATURES_T].copy()

In [11]:
from xgboost import XGBClassifier
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 5,
    'colsample_bytree': 0.5,
    'subsample': 0.8,
    'n_estimators': 10000,
    'learning_rate': 0.01,
    'early_stopping_rounds': 100,
    'random_state': 42,
    'n_jobs': -1,
    'device': 'cuda',
    'enable_categorical': True,
}

skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = XGBClassifier(**params)

    model.fit(X_train, y_train,
             eval_set=[(X_val, y_val)],
             verbose=1000)

    val_preds = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = val_preds

    fold_score = roc_auc_score(y_val, val_preds)
    print(f'FOLD {fold} AUC: {fold_score:.4f}')
    test_preds +=  model.predict_proba(test_n)[:, 1] / config.N_FOLDS

overall_auc = roc_auc_score(y, oof_preds)
print('='*30)
print(f"Overall OOF AUC: {overall_auc:.4f}")
print('='*30)

[0]	validation_0-auc:0.60746
[1000]	validation_0-auc:0.71957
[2000]	validation_0-auc:0.72384
[3000]	validation_0-auc:0.72574
[4000]	validation_0-auc:0.72684
[5000]	validation_0-auc:0.72742
[6000]	validation_0-auc:0.72775
[7000]	validation_0-auc:0.72792
[8000]	validation_0-auc:0.72807
FOLD 0 AUC: 0.7281
[0]	validation_0-auc:0.61024
[1000]	validation_0-auc:0.71757
[2000]	validation_0-auc:0.72200
[3000]	validation_0-auc:0.72381
[4000]	validation_0-auc:0.72485
[5000]	validation_0-auc:0.72547
[6000]	validation_0-auc:0.72588
[7000]	validation_0-auc:0.72616
[8000]	validation_0-auc:0.72634
[8776]	validation_0-auc:0.72640
FOLD 1 AUC: 0.7264
[0]	validation_0-auc:0.60667
[1000]	validation_0-auc:0.71817
[2000]	validation_0-auc:0.72233
[3000]	validation_0-auc:0.72446
[4000]	validation_0-auc:0.72555
[5000]	validation_0-auc:0.72619
[6000]	validation_0-auc:0.72661
[7000]	validation_0-auc:0.72689
[8000]	validation_0-auc:0.72706
[8131]	validation_0-auc:0.72706
FOLD 2 AUC: 0.7271
[0]	validation_0-auc:0.6

In [12]:
submission[config.TARGET] = test_preds
submission.to_csv(f'submission_cv_{overall_auc}.csv', index=False)